# Контроль качества, интеграция и первичная визуализация

Цель: получить чистую таблицу уровня обращения, не размножив строки при объединении источников.

> Работайте последовательно и сохраняйте результаты в `outputs/`. После каждого раздела сверяйтесь с контрольной точкой. Полные решения в студенческой версии не приводятся.

> **Место в производственном маршруте:** 2 из 9  
> **Ориентир очного занятия:** 35–95 минут  
> **Режим:** Основной практический маршрут  
> **Выход этапа:** Чистые витрины и первичная визуализация

Студенческая версия содержит задания и контрольные точки без полного решения.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "tickets.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не найдена папка проекта. Убедитесь, что notebook находится внутри распакованного комплекта."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [OUTPUT_DIR, CHART_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Корень проекта:", PROJECT_ROOT)
print("Исходные данные:", RAW_DIR)
print("Результаты:", OUTPUT_DIR)

In [ ]:
required_files = {
    "tickets": RAW_DIR / "tickets.csv",
    "events": RAW_DIR / "ticket_events.csv",
    "capacity": RAW_DIR / "team_capacity_daily.csv",
    "calendar": RAW_DIR / "calendar_events.csv",
    "load_log": RAW_DIR / "load_log.csv",
    "teams": RAW_DIR / "teams.xlsx",
    "regions": RAW_DIR / "regions.json",
}

missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Не найдены файлы:\n" + "\n".join(missing))

raw_tickets = pd.read_csv(required_files["tickets"])
raw_events = pd.read_csv(required_files["events"])
raw_capacity = pd.read_csv(required_files["capacity"])
calendar = pd.read_csv(required_files["calendar"])
load_log = pd.read_csv(required_files["load_log"])
teams = pd.read_excel(required_files["teams"])
regions = pd.read_json(required_files["regions"])

print("Загружено:")
for name, frame in {
    "tickets": raw_tickets,
    "ticket_events": raw_events,
    "team_capacity_daily": raw_capacity,
    "calendar_events": calendar,
    "load_log": load_log,
    "teams": teams,
    "regions": regions,
}.items():
    print(f"- {name}: {frame.shape[0]:,} строк × {frame.shape[1]} столбцов")

## Задание 1. Зафиксируйте исходные объёмы

Создайте таблицу с названием источника, числом строк и столбцов. Контроль: `tickets.csv` содержит 24 240 строк.

In [ ]:
# TODO: соберите raw_overview и выведите его

## Задание 2. Очистите `tickets`

1. Преобразуйте временные поля через `pd.to_datetime(..., errors="coerce", utc=True)`.
2. Оставьте последнюю версию каждого `ticket_id` по `source_updated_at`.
3. Нормализуйте текстовые категории.
4. Восстановите `region_id` по справочнику команды.
5. Замените отрицательный `first_response_min` и некорректный `csat` на пропуски.
6. Исключите фатальные нарушения.

Контроль: 23 922 строки, из них 23 000 с известным `sla_breached`.

In [ ]:
tickets = raw_tickets.copy()
# TODO: выполните очистку

print("Строк после очистки:", len(tickets))

## Задание 3. Очистите и агрегируйте журнал событий

Сначала удалите дубли и ошибочные события. Затем получите одну строку на `ticket_id` с количеством событий, переводов, эскалаций и временем первого назначения.

Контроль: после очистки — 144 800 событий; `ticket_id` в агрегате уникален.

In [ ]:
events = raw_events.copy()
# TODO: очистите события и создайте event_features

## Задание 4. Восстановите сетку мощности команд

Создайте полный набор `546 дней × 32 команды`. Добавьте `is_reconstructed`.

Контроль: 17 472 строки, 120 восстановленных комбинаций.

In [ ]:
capacity = raw_capacity.copy()
# TODO: удалите дубли, исправьте нарушения и восстановите полную сетку

## Задание 5. Соберите `analysis_base`

Присоединяйте агрегат событий, справочники, календарь и мощность. Используйте `validate`.

Контроль: после соединений число строк должно остаться 23 922.

In [ ]:
# TODO: соберите analysis_base

## Задание 6. Постройте первичные графики

Минимум:

- динамика обращений по дням;
- распределение времени решения;
- обращения по каналам;
- тепловая карта «день недели × час»;
- доля SLA-нарушений по регионам.

In [ ]:
# TODO: создайте daily_demand и графики

## Задание 7. Сохраните результаты

Сохраните чистые таблицы и `data_quality_report.csv`.

In [ ]:
# TODO: сохраните outputs/tickets_clean.csv, ticket_event_features.csv,
# team_capacity_clean.csv, daily_demand.csv, analysis_base.csv и data_quality_report.csv

## Самопроверка

- [ ] ключ `ticket_id` уникален в чистой карточке;
- [ ] события агрегированы до одной строки на обращение;
- [ ] соединения не увеличили число строк;
- [ ] даты образуют непрерывный диапазон;
- [ ] графики имеют подписи и аналитический вывод.